# Training Sequential Type-First Disease Classifier

## Paper: Feature-Driven Priority Queuing

This notebook trains a MobileNet-based disease classifier using the **type-first (sequential) approach** described in the paper. The model learns to predict disease posteriors for each of the T=14 disease types present in the NIH ChestX-ray14 dataset.

### Key Components:
- **Dataset**: NIH ChestX-ray14 (>110K images, 30 disease labels)
- **Model**: MobileNet pre-trained on ImageNet
- **Task**: Disease classification (type-first approach)
- **Output**: T=14 disease probability scores per image
- **Queuing**: N=4 priority queues based on disease type

### Approach:
In the type-first sequential approach, each image is assigned to exactly one type based on the highest-priority disease present (using the ranking in Table 2). The classifier then predicts posterior probabilities for all 13 diseases (excluding 'No Finding').

## Section 1: Setup & Configuration

Import required libraries and define configuration parameters for data processing and model training.

In [ ]:
import numpy as np
import pandas as pd
import os
from glob import glob
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications.mobilenet import MobileNet
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras import optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, confusion_matrix, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
import pickle
from itertools import chain
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(2018)
tf.random.set_seed(2018)

### Configuration Parameters

These parameters control data paths, image preprocessing, model architecture, and training.

In [ ]:
# ============================================================================
# DATA PATHS
# ============================================================================
DATA_DIR = '../data'  # Path to NIH ChestX-ray14 dataset
IMAGES_DIR = os.path.join(DATA_DIR, 'images')  # Directory containing chest X-ray images
CSV_FILE = os.path.join(DATA_DIR, 'Data_Entry_2017.csv')  # Metadata CSV
OUTPUT_DIR = '../model_weights'  # Directory to save trained weights
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# IMAGE PREPROCESSING
# ============================================================================
IMG_SIZE = (512, 512)  # Input image resolution
COLOR_MODE = 'rgb'  # Color mode (rgb for 3-channel images)
CHANNELS = 3  # Number of color channels
BATCH_SIZE = 32  # Batch size for training

# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================
LEARNING_RATE = 0.0005  # Adam optimizer learning rate
DROPOUT_RATE = 0.5  # Dropout rate for regularization
HIDDEN_UNITS = 512  # Number of units in dense layers

# ============================================================================
# TRAINING
# ============================================================================
STEPS_PER_EPOCH = 1000  # Training steps per epoch
EPOCHS = 100  # Maximum number of epochs
EARLY_STOP_PATIENCE = 10  # Patience for early stopping

# ============================================================================
# DATA FILTERING
# ============================================================================
MIN_CASES = 1000  # Minimum number of cases to include a disease label
RANDOM_STATE = 2018  # Random seed for reproducibility

# ============================================================================
# PAPER PARAMETERS
# ============================================================================
T = 14  # Number of disease types in the dataset
N = 4   # Number of priority queues for feature-driven prioritization

print(f"Configuration:")
print(f"  Image size: {IMG_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Disease types (T): {T}")
print(f"  Priority queues (N): {N}")

## Section 2: Data Loading & Preprocessing

Load the ChestX-ray14 dataset from the CSV metadata file and match image paths to diagnostic labels.

In [ ]:
# Load the metadata CSV file
print("Loading ChestX-ray14 metadata...")
all_xray_df = pd.read_csv(CSV_FILE)
print(f"Loaded {len(all_xray_df)} images from metadata")
print(f"\nColumns: {all_xray_df.columns.tolist()}")
print(f"\nFirst row:")
print(all_xray_df.iloc[0])

In [ ]:
# Extract all disease labels from the dataset
print("Extracting disease labels...")

# Split 'Finding Labels' field by '|' and collect all unique labels
all_labels_raw = list(chain(*all_xray_df['Finding Labels'].map(
    lambda x: x.split('|')
).tolist()))

all_labels = np.unique(all_labels_raw)
print(f"Total unique labels: {len(all_labels)}")
print(f"Labels: {all_labels}")

In [ ]:
# Remove 'No Finding' from active disease labels
all_labels = [c for c in all_labels if c != 'No Finding']
print(f"Disease labels (excluding 'No Finding'): {len(all_labels)}")
print(f"Labels: {all_labels}")

In [ ]:
# Create binary columns for each disease label
print(f"\nCreating binary label columns...")
for label in all_labels:
    all_xray_df[label] = all_xray_df['Finding Labels'].map(
        lambda finding: 1 if label in finding else 0
    )

# Filter to labels with at least MIN_CASES positive examples
print(f"Filtering to labels with >= {MIN_CASES} cases...")
label_counts = {label: all_xray_df[label].sum() for label in all_labels}
all_labels_filtered = [label for label in all_labels if label_counts[label] >= MIN_CASES]

print(f"\nDisease label statistics:")
for label in sorted(all_labels_filtered, key=lambda x: label_counts[x], reverse=True):
    print(f"  {label}: {label_counts[label]} cases")

all_labels = all_labels_filtered
print(f"\nFinal disease labels (T={len(all_labels)}): {all_labels}")

In [ ]:
# Construct full image paths
print("Constructing image file paths...")
all_xray_df['path'] = all_xray_df['Image Index'].map(
    lambda img_idx: os.path.join(IMAGES_DIR, img_idx)
)

# Filter to images that exist in the file system
all_xray_df['file_exists'] = all_xray_df['path'].map(
    lambda x: os.path.isfile(x)
)

n_before = len(all_xray_df)
all_xray_df = all_xray_df[all_xray_df['file_exists']]
n_after = len(all_xray_df)

print(f"Images with existing files: {n_after}/{n_before}")
print(f"Sample image paths:")
for path in all_xray_df['path'].head(3):
    print(f"  {path}")

## Section 3: Type Assignment (Table 2 - Disease Ranking)

Assign each image to one of T disease types based on the highest-priority (lowest-rank) disease present. This implements the disease priority ranking from Table 2 of the paper.

In [ ]:
def assign_image_type(findings):
    """
    Assign image type based on highest-priority disease present.
    
    The type is determined by the most urgent (lowest rank number) disease
    using the ranking from Table 2 of the paper:
    
    Rank  Disease              Priority
    ----  -------              --------
    1     Pneumothorax         Most urgent
    2     Emphysema
    3     Pneumonia
    4     Edema
    5     Consolidation
    6     Effusion
    7     Infiltration
    8     Atelectasis
    9     Cardiomegaly
    10    Pleural_Thickening
    11    Fibrosis
    12    Mass
    13    Nodule
    14    No Finding           Least urgent
    
    Args:
        findings (str): Pipe-separated string of disease labels
    
    Returns:
        int: Rank of the highest-priority disease (1-14)
    """
    disease_rank = {
        'Pneumothorax': 1,
        'Emphysema': 2,
        'Pneumonia': 3,
        'Edema': 4,
        'Consolidation': 5,
        'Effusion': 6,
        'Infiltration': 7,
        'Atelectasis': 8,
        'Cardiomegaly': 9,
        'Pleural_Thickening': 10,
        'Fibrosis': 11,
        'Mass': 12,
        'Nodule': 13,
        'No Finding': 14
    }
    
    labels = findings.split('|')
    ranks = [disease_rank.get(l, 14) for l in labels]
    return min(ranks)


# Assign type to each image
print("Assigning image types based on disease priority ranking...")
all_xray_df['image_type'] = all_xray_df['Finding Labels'].map(assign_image_type)

# Map rank to disease name for interpretability
rank_to_disease = {
    1: 'Pneumothorax', 2: 'Emphysema', 3: 'Pneumonia', 4: 'Edema',
    5: 'Consolidation', 6: 'Effusion', 7: 'Infiltration', 8: 'Atelectasis',
    9: 'Cardiomegaly', 10: 'Pleural_Thickening', 11: 'Fibrosis',
    12: 'Mass', 13: 'Nodule', 14: 'No Finding'
}
all_xray_df['type_name'] = all_xray_df['image_type'].map(rank_to_disease)

print(f"\nType distribution:")
type_counts = all_xray_df['image_type'].value_counts().sort_index()
for type_idx in sorted(all_xray_df['image_type'].unique()):
    disease_name = rank_to_disease[type_idx]
    count = type_counts[type_idx]
    pct = 100 * count / len(all_xray_df)
    print(f"  Type {type_idx:2d} ({disease_name:20s}): {count:6d} images ({pct:5.1f}%)")

## Section 4: Train/Validation/Test Split & Class Balancing

Split data into 50% training, 25% validation, and 25% test sets. Apply RandomOverSampler to the training set to balance class distribution across disease types.

In [ ]:
# First split: separate test set (25%)
print("Splitting data into train/validation/test sets...")
train_valid_df, test_df = train_test_split(
    all_xray_df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=all_xray_df['image_type']
)

# Second split: split remaining into train/validation (50/25)
train_df, valid_df = train_test_split(
    train_valid_df,
    test_size=1/3,  # 1/3 of 75% = 25%
    random_state=RANDOM_STATE,
    stratify=train_valid_df['image_type']
)

print(f"\nData split:")
print(f"  Training set:   {len(train_df):6d} images ({100*len(train_df)/len(all_xray_df):.1f}%)")
print(f"  Validation set: {len(valid_df):6d} images ({100*len(valid_df)/len(all_xray_df):.1f}%)")
print(f"  Test set:       {len(test_df):6d} images ({100*len(test_df)/len(all_xray_df):.1f}%)")
print(f"  Total:          {len(all_xray_df):6d} images")

In [ ]:
# Apply RandomOverSampler to balance training set
print("\nApplying RandomOverSampler to training set...")
print(f"Training set before balancing:")
print(train_df['image_type'].value_counts().sort_index())

# Create feature matrix (dummy features) for resampling
X_train = np.arange(len(train_df)).reshape(-1, 1)
y_train = train_df['image_type'].values

# Apply random oversampling
ros = RandomOverSampler(random_state=RANDOM_STATE)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

# Get resampled indices and apply to dataframe
train_indices = X_resampled.flatten()
train_df = train_df.iloc[train_indices].reset_index(drop=True)

print(f"\nTraining set after balancing:")
print(train_df['image_type'].value_counts().sort_index())
print(f"New training set size: {len(train_df)}")

## Section 5: Image Data Generator

Create an ImageDataGenerator with data augmentation and normalization to improve generalization and prevent overfitting.

In [ ]:
# Define image data generator with augmentation
print("Creating ImageDataGenerator with augmentation...")

core_idg = ImageDataGenerator(
    samplewise_center=True,  # Normalize each image independently
    samplewise_std_normalization=True,  # Standardize pixel values
    horizontal_flip=True,  # Random horizontal flips
    vertical_flip=False,  # No vertical flips (inappropriate for medical images)
    height_shift_range=0.05,  # Random height shifts
    width_shift_range=0.1,  # Random width shifts
    rotation_range=5,  # Random rotations (small range for medical data)
    shear_range=0.1,  # Shear transformations
    fill_mode='reflect',  # Fill missing pixels by reflection
    zoom_range=0.15  # Random zoom
)

print("ImageDataGenerator created successfully")

In [ ]:
# Create data generators for training, validation, and test
print("Creating data generators for each split...")

# Training generator (with augmentation)
train_gen = core_idg.flow_from_dataframe(
    train_df,
    x_col='path',
    y_col=all_labels,
    target_size=IMG_SIZE,
    color_mode=COLOR_MODE,
    classes=all_labels,
    class_mode='raw',
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Validation generator (no augmentation)
valid_idg = ImageDataGenerator(
    samplewise_center=True,
    samplewise_std_normalization=True
)

valid_gen = valid_idg.flow_from_dataframe(
    valid_df,
    x_col='path',
    y_col=all_labels,
    target_size=IMG_SIZE,
    color_mode=COLOR_MODE,
    classes=all_labels,
    class_mode='raw',
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Test generator (no augmentation)
test_idg = ImageDataGenerator(
    samplewise_center=True,
    samplewise_std_normalization=True
)

test_gen = test_idg.flow_from_dataframe(
    test_df,
    x_col='path',
    y_col=all_labels,
    target_size=IMG_SIZE,
    color_mode=COLOR_MODE,
    classes=all_labels,
    class_mode='raw',
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"Training generator: {len(train_gen)} batches")
print(f"Validation generator: {len(valid_gen)} batches")
print(f"Test generator: {len(test_gen)} batches")

## Section 6: Model Architecture

Build the MobileNet-based disease classifier. This model serves as the type predictor in the type-first sequential approach, outputting posterior probabilities for each of the T disease types.

In [ ]:
def build_sequential_model(img_size, channels, num_diseases, hidden_units=HIDDEN_UNITS, dropout_rate=DROPOUT_RATE):
    """
    Build MobileNet-based disease classifier for type-first approach.
    
    Architecture:
    - MobileNet base (pre-trained on ImageNet, all layers trainable)
    - Global Average Pooling (spatial dimension reduction)
    - Dropout (0.5) for regularization
    - Dense layer (512 units) with ReLU activation
    - Dropout (0.5) for regularization
    - Output layer (num_diseases units) with sigmoid activation
    
    The sigmoid activation enables multi-label classification (images can have
    multiple diseases present simultaneously).
    
    Args:
        img_size (tuple): Input image size (height, width)
        channels (int): Number of color channels (3 for RGB)
        num_diseases (int): Number of disease labels to predict
        hidden_units (int): Number of units in dense layer (default: 512)
        dropout_rate (float): Dropout rate (default: 0.5)
    
    Returns:
        Sequential: Compiled Keras model
    """
    model = Sequential([
        # Load MobileNet base model (pre-trained on ImageNet)
        MobileNet(
            input_shape=(*img_size, channels),
            include_top=False,  # Remove classification head
            weights='imagenet'  # Use ImageNet pre-training
        ),
        
        # Global average pooling to reduce spatial dimensions
        GlobalAveragePooling2D(),
        
        # First dropout layer
        Dropout(dropout_rate),
        
        # Dense layer with ReLU activation
        Dense(hidden_units, activation='relu'),
        
        # Second dropout layer
        Dropout(dropout_rate),
        
        # Output layer: sigmoid for multi-label classification
        # Each neuron predicts probability of one disease being present
        Dense(num_diseases, activation='sigmoid')
    ])
    
    return model


# Build the model
print("Building MobileNet disease classifier...")
model = build_sequential_model(
    img_size=IMG_SIZE,
    channels=CHANNELS,
    num_diseases=len(all_labels),
    hidden_units=HIDDEN_UNITS,
    dropout_rate=DROPOUT_RATE
)

print(f"\nModel Summary:")
model.summary()

## Section 7: Model Compilation & Training

Compile the model with binary cross-entropy loss (appropriate for multi-label classification) and train with callbacks for checkpointing and early stopping.

In [ ]:
# Compile the model
print("Compiling model...")

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',  # Multi-label classification loss
    metrics=['binary_accuracy', 'mae']  # Binary accuracy and mean absolute error
)

print("Model compiled successfully")

In [ ]:
# Define callbacks
print("Setting up training callbacks...")

# ModelCheckpoint: save best weights based on validation loss
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    os.path.join(OUTPUT_DIR, 'sequential_weights.hdf5'),
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# EarlyStopping: stop training if validation loss plateaus
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOP_PATIENCE,
    verbose=1,
    restore_best_weights=True
)

# ReduceLROnPlateau: reduce learning rate if validation loss plateaus
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

callbacks_list = [checkpoint, early_stop, reduce_lr]
print("Callbacks configured")

In [ ]:
# Train the model
print("Starting training...")
print(f"Training configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Steps per epoch: {STEPS_PER_EPOCH}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Disease labels: {len(all_labels)}")

history = model.fit(
    train_gen,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=valid_gen,
    validation_steps=len(valid_gen),
    epochs=EPOCHS,
    callbacks=callbacks_list,
    verbose=1
)

print("Training completed")

## Section 8: Training History Visualization

Plot training and validation losses to visualize model convergence.

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Model Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Binary Accuracy
axes[1].plot(history.history['binary_accuracy'], label='Training Accuracy')
axes[1].plot(history.history['val_binary_accuracy'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Binary Accuracy')
axes[1].set_title('Model Binary Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# MAE
axes[2].plot(history.history['mae'], label='Training MAE')
axes[2].plot(history.history['val_mae'], label='Validation MAE')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Mean Absolute Error')
axes[2].set_title('Model MAE')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Training history saved to {os.path.join(OUTPUT_DIR, 'training_history.png')}")

## Section 9: Model Evaluation on Test Set

Evaluate the trained model on the test set and generate predictions for ROC analysis.

In [ ]:
# Load best weights
print("Loading best model weights...")
model.load_weights(os.path.join(OUTPUT_DIR, 'sequential_weights.hdf5'))
print("Best weights loaded")

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_loss, test_binary_acc, test_mae = model.evaluate(test_gen, verbose=0)

print(f"\nTest Set Performance:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Binary Accuracy: {test_binary_acc:.4f}")
print(f"  Mean Absolute Error: {test_mae:.4f}")

In [ ]:
# Generate predictions on test set
print("Generating test set predictions...")
test_predictions = model.predict(test_gen, verbose=1)
test_labels = test_gen.classes  # True labels from generator

print(f"Predictions shape: {test_predictions.shape}")
print(f"Labels shape: {test_labels.shape}")
print(f"Sample predictions (first 5 images, first 5 diseases):")
print(test_predictions[:5, :5])

## Section 10: ROC Curves & AUC Analysis (Figure 5)

Generate ROC curves for each disease label and compute per-disease and macro-average AUC scores.

In [ ]:
# Compute ROC curves and AUC for each disease
print("Computing ROC curves and AUC scores...")

auc_scores = {}
roc_curves = {}

for i, label in enumerate(all_labels):
    fpr, tpr, _ = roc_curve(test_labels[:, i], test_predictions[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores[label] = roc_auc
    roc_curves[label] = (fpr, tpr)

# Sort by AUC score
sorted_labels = sorted(auc_scores.items(), key=lambda x: x[1], reverse=True)

print(f"\nAUC Scores by Disease:")
print(f"{'Disease':<25} {'AUC':<8}")
print("-" * 33)
for label, auc_score in sorted_labels:
    print(f"{label:<25} {auc_score:.4f}")

# Macro-average AUC
macro_auc = np.mean(list(auc_scores.values()))
print(f"\nMacro-average AUC: {macro_auc:.4f}")

In [ ]:
# Plot ROC curves (Figure 5)
print("Plotting ROC curves...")

# Plot for top 6 diseases by AUC
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (label, _) in enumerate(sorted_labels[:6]):
    fpr, tpr = roc_curves[label]
    auc_score = auc_scores[label]
    
    axes[idx].plot(fpr, tpr, lw=2, 
                   label=f'ROC curve (AUC = {auc_score:.3f})')
    axes[idx].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    axes[idx].set_xlabel('False Positive Rate')
    axes[idx].set_ylabel('True Positive Rate')
    axes[idx].set_title(f'{label}')
    axes[idx].legend(loc='lower right')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_curves_top6.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"ROC curves saved to {os.path.join(OUTPUT_DIR, 'roc_curves_top6.png')}")

In [ ]:
# Plot all AUC scores
fig, ax = plt.subplots(figsize=(12, 6))

labels_sorted = [label for label, _ in sorted_labels]
auc_values_sorted = [auc_score for _, auc_score in sorted_labels]

bars = ax.barh(range(len(labels_sorted)), auc_values_sorted)
ax.set_yticks(range(len(labels_sorted)))
ax.set_yticklabels(labels_sorted)
ax.set_xlabel('AUC Score')
ax.set_title('Per-Disease AUC Scores')
ax.set_xlim([0, 1])

# Color bars by AUC
for bar, auc_val in zip(bars, auc_values_sorted):
    if auc_val >= 0.9:
        bar.set_color('green')
    elif auc_val >= 0.8:
        bar.set_color('yellow')
    else:
        bar.set_color('red')

# Add value labels
for i, (label, auc_val) in enumerate(sorted_labels):
    ax.text(auc_val + 0.02, i, f'{auc_val:.3f}', va='center')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'auc_scores.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC plot saved to {os.path.join(OUTPUT_DIR, 'auc_scores.png')}")

## Section 11: Optimal Thresholds (Youden Index)

Compute optimal classification thresholds for each disease using the Youden Index (sensitivity + specificity - 1).

In [ ]:
def compute_youden_threshold(y_true, y_pred):
    """
    Compute optimal classification threshold using Youden Index.
    
    Youden Index = Sensitivity + Specificity - 1
    
    This metric measures how well a threshold separates positive and negative
    cases, with higher values indicating better separation.
    
    Args:
        y_true (np.array): True binary labels
        y_pred (np.array): Predicted probabilities
    
    Returns:
        float: Optimal threshold that maximizes Youden Index
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    youden_index = tpr - fpr
    optimal_idx = np.argmax(youden_index)
    optimal_threshold = thresholds[optimal_idx]
    return optimal_threshold


# Compute optimal thresholds for each disease
print("Computing optimal thresholds using Youden Index...")

optimal_thresholds = {}
for i, label in enumerate(all_labels):
    threshold = compute_youden_threshold(test_labels[:, i], test_predictions[:, i])
    optimal_thresholds[label] = threshold

print(f"\nOptimal Thresholds by Disease:")
print(f"{'Disease':<25} {'Threshold':<12}")
print("-" * 37)
for label in all_labels:
    print(f"{label:<25} {optimal_thresholds[label]:.4f}")

In [ ]:
# Visualize threshold distribution
fig, ax = plt.subplots(figsize=(10, 6))

thresholds_list = list(optimal_thresholds.values())
labels_list = list(optimal_thresholds.keys())

ax.bar(range(len(labels_list)), thresholds_list)
ax.set_xticks(range(len(labels_list)))
ax.set_xticklabels(labels_list, rotation=45, ha='right')
ax.set_ylabel('Optimal Threshold')
ax.set_title('Optimal Classification Thresholds (Youden Index)')
ax.axhline(y=0.5, color='r', linestyle='--', label='Default threshold (0.5)')
ax.set_ylim([0, 1])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'optimal_thresholds.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Thresholds plot saved to {os.path.join(OUTPUT_DIR, 'optimal_thresholds.png')}")

## Section 12: Save Model, Weights, & Thresholds

Save the trained model architecture, weights, disease thresholds, and metadata for later use in the feature-driven priority queuing pipeline.

In [ ]:
# Save model architecture as JSON
print("Saving model architecture...")
model_json = model.to_json()
with open(os.path.join(OUTPUT_DIR, 'sequential_model.json'), 'w') as json_file:
    json_file.write(model_json)
print(f"Model architecture saved to {os.path.join(OUTPUT_DIR, 'sequential_model.json')}")

In [ ]:
# Save model using TensorFlow SavedModel format
print("\nSaving model in SavedModel format...")
model_dir = os.path.join(OUTPUT_DIR, 'sequential_model_savedmodel')
model.save(model_dir)
print(f"Model saved to {model_dir}")

In [ ]:
# Save optimal thresholds and metadata
print("\nSaving thresholds and metadata...")

metadata = {
    'disease_labels': all_labels,
    'optimal_thresholds': optimal_thresholds,
    'auc_scores': auc_scores,
    'macro_auc': macro_auc,
    'test_loss': float(test_loss),
    'test_binary_accuracy': float(test_binary_acc),
    'test_mae': float(test_mae),
    'img_size': IMG_SIZE,
    'color_mode': COLOR_MODE,
    'channels': CHANNELS,
    'num_diseases': len(all_labels),
    'T': T,
    'N': N,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'epochs_trained': len(history.history['loss']),
}

metadata_path = os.path.join(OUTPUT_DIR, 'sequential_metadata.pkl')
with open(metadata_path, 'wb') as f:
    pickle.dump(metadata, f)
print(f"Metadata saved to {metadata_path}")

In [ ]:
# Save results summary
print("\nSaving results summary...")

summary_text = f"""
=============================================================================
SEQUENTIAL TYPE-FIRST DISEASE CLASSIFIER - RESULTS SUMMARY
=============================================================================

DATASET CONFIGURATION
  - Total images: {len(all_xray_df)}
  - Disease labels: {len(all_labels)}
  - Training set: {len(train_df)} images
  - Validation set: {len(valid_df)} images
  - Test set: {len(test_df)} images

MODEL ARCHITECTURE
  - Base model: MobileNet (ImageNet pre-trained)
  - Input size: {IMG_SIZE[0]}x{IMG_SIZE[1]} pixels
  - Color mode: {COLOR_MODE}
  - Hidden layer units: {HIDDEN_UNITS}
  - Dropout rate: {DROPOUT_RATE}
  - Output units: {len(all_labels)} (one per disease)
  - Output activation: sigmoid (multi-label classification)

TRAINING CONFIGURATION
  - Optimizer: Adam
  - Learning rate: {LEARNING_RATE}
  - Loss: binary_crossentropy
  - Batch size: {BATCH_SIZE}
  - Steps per epoch: {STEPS_PER_EPOCH}
  - Max epochs: {EPOCHS}
  - Actual epochs trained: {len(history.history['loss'])}

TEST SET PERFORMANCE
  - Loss: {test_loss:.4f}
  - Binary Accuracy: {test_binary_acc:.4f}
  - Mean Absolute Error: {test_mae:.4f}
  - Macro-average AUC: {macro_auc:.4f}

PER-DISEASE AUC SCORES
"""

for label, auc_score in sorted_labels:
    summary_text += f"  - {label:<25}: {auc_score:.4f}\n"

summary_text += f"""
PAPER PARAMETERS
  - T (disease types): {T}
  - N (priority queues): {N}

OUTPUT FILES
  - Model weights (HDF5): sequential_weights.hdf5
  - Model architecture (JSON): sequential_model.json
  - Model (SavedModel): sequential_model_savedmodel/
  - Metadata (pickle): sequential_metadata.pkl
  - Training history plot: training_history.png
  - ROC curves (top 6): roc_curves_top6.png
  - AUC scores plot: auc_scores.png
  - Optimal thresholds plot: optimal_thresholds.png

=============================================================================
"""

summary_path = os.path.join(OUTPUT_DIR, 'RESULTS_SUMMARY.txt')
with open(summary_path, 'w') as f:
    f.write(summary_text)

print(summary_text)
print(f"Summary saved to {summary_path}")

## Section 13: Model Testing & Inference

Demonstrate model inference on sample test images and show predictions.

In [ ]:
# Reload best model for inference
print("Loading best model for inference testing...")
model.load_weights(os.path.join(OUTPUT_DIR, 'sequential_weights.hdf5'))
print("Model loaded successfully")

In [ ]:
# Test inference on sample images
print("\nTesting model inference on sample images...")

# Get a batch from test generator
sample_images, sample_labels = next(iter(test_gen))
print(f"Sample batch size: {sample_images.shape[0]}")
print(f"Image shape: {sample_images.shape[1:]}")

# Make predictions
sample_predictions = model.predict(sample_images, verbose=0)

# Display results for first 3 images
print(f"\nSample Predictions (first 3 images):")
print("=" * 80)

for idx in range(min(3, sample_images.shape[0])):
    print(f"\nImage {idx + 1}:")
    print(f"  True labels: {[all_labels[i] for i in range(len(all_labels)) if sample_labels[idx, i] == 1]}")
    print(f"  Predictions:")
    
    # Get predictions sorted by confidence
    pred_dict = {all_labels[i]: sample_predictions[idx, i] for i in range(len(all_labels))}
    sorted_preds = sorted(pred_dict.items(), key=lambda x: x[1], reverse=True)
    
    for disease, prob in sorted_preds[:5]:
        threshold = optimal_thresholds[disease]
        predicted = "Yes" if prob >= threshold else "No"
        print(f"    {disease:<25}: {prob:.4f} (threshold: {threshold:.4f}, prediction: {predicted})")

## Final Summary

The sequential type-first disease classifier has been successfully trained on the NIH ChestX-ray14 dataset. The model uses a pre-trained MobileNet architecture to predict posterior probabilities for 13 disease types.

### Key Results:
- **Test Loss**: {:.4f}
- **Test Accuracy**: {:.4f}
- **Macro-average AUC**: {:.4f}

### Trained Models & Artifacts:
All trained weights, model architecture, optimal thresholds, and evaluation metrics have been saved to the `weights/` directory and are ready for use in the feature-driven priority queuing pipeline.

### Next Steps:
1. Use the trained model to generate disease posterior predictions on the full dataset
2. Apply the optimal thresholds to convert probabilities to binary predictions
3. Assign images to priority queues based on disease type and posterior probabilities
4. Evaluate the feature-driven priority queuing performance on downstream tasks
""".format(test_loss, test_binary_acc, macro_auc)